# Kadastra BO2 — Preprocessing Pipeline

**Project:** Kadastra — AI-powered real estate platform for the Tunisian market  
**Business Objective:** BO2 — Property price estimation & market trend monitoring  
**Author:** Kadastra BO2 Team  

---

## Overview

This notebook implements the full preprocessing pipeline for BO2. Its sole responsibility is to transform the raw scraped dataset into clean, model-ready CSV files — one per property sub-type.

The raw data was scraped from two Tunisian real estate portals — **Mubawab** and **Tayara** — and merged into a single file (`final_dfV2-2.csv`, 30,186 rows × 24 columns). The two sources use incompatible formats for property type, price, and location, which requires careful normalisation before any modelling can begin.

### Pipeline structure

| Step | Cell | Description |
|------|------|-------------|
| 0 | Cell 1 | Configuration & imports |
| 0 | Cell 2 | Upload & load raw data |
| 1 | Cell 3 | Transaction & category normalisation |
| 1b | Cell 4 | Immeuble (whole-building) detection |
| 2 | Cell 5 | S+N notation & floor extraction |
| 3 | Cell 6 | Price cleaning & log transform |
| 4 | Cell 7 | Surface cleaning & prix/m² sanity check |
| 5 | Cell 8 | Pieces/chambres consistency repair |
| 6 | Cell 9 | Null flags (no imputation) |
| 7 | Cell 10 | Feature engineering (spatial, temporal, NLP) |
| 7b | Cell 11 | Zone-relative price floor |
| 7c | Cell 12 | Terrain double-null removal |
| 8–9 | Cell 13 | Sub-dataset split & feature manifests |
| 10 | Cell 14 | Save & download outputs |

### Design principles

1. **Separation of concerns** — preprocessing never leaks label information. All zone-based statistics computed during feature engineering (Cell 10) use the full dataset; fold-wise statistics are computed in the modelling notebook.
2. **Auditability** — every row-dropping step is logged via the `log()` function, producing a full audit trail in `pipeline_report.txt`.
3. **Separate models per sub-type** — price drivers differ fundamentally across property types (appartement: surface + floor + amenities; terrain: surface + zone only). A single combined model would produce a compromise poor for all types.
4. **No global imputation** — missing values are flagged (`_was_null` columns) but not filled here. Imputation uses training-fold statistics in the modelling notebook to prevent leakage.

## Cell 1 — Configuration & Imports

All tuneable thresholds are centralised in a single `CONFIG` dictionary. This is intentional: when BO1 delivers more data or when we want to experiment with different price ranges, we change one value here rather than hunting through ten cells.

Key thresholds and their rationale:
- **`sale_price_min = 5,000 DT`** — global floor; revised upward to 30,000 DT specifically for `appartement` vente in Cell 6 (see Step 3)
- **`prix_m2_min/max = [200, 15,000] DT/m²`** — any listing outside this range has either a data entry error or belongs to a niche segment (e.g. a symbolic 1 DT/m² land gift, or a 20,000 DT/m² penthouse) that would distort the model
- **`min_rows_to_train = 200`** — sub-datasets below this size are isolated into `predict_pool` for future use; training on fewer than 200 rows produces unreliable CV estimates
- **Immeuble thresholds** — the structural signals (12+ rooms, 500+ m², 2M+ DT) that distinguish a whole building from a single unit

In [ ]:
import pandas as pd
import numpy as np
import re, os, io
from datetime import datetime

CONFIG = {
    'sale_price_min':          5_000,
    'sale_price_max':      8_000_000,
    'rent_price_min':            200,
    'rent_price_max':         20_000,
    'surface_min':                10,
    'appt_surface_max':          600,
    'maison_surface_max':      1_500,
    'terrain_surface_max':    50_000,
    'local_surface_max':       2_000,
    'immeuble_surface_max':    5_000,
    'prix_m2_min':               200,
    'prix_m2_max':            15_000,
    'sn_to_pieces':   {0:1, 1:2, 2:3, 3:4, 4:5, 5:6, 6:7},
    'min_rows_to_train':         200,
    'immeuble_pieces_min':        12,
    'immeuble_chambres_min':       8,
    'immeuble_surface_min':      500,
    'immeuble_price_vente':  2_000_000,
    'immeuble_price_location':  8_000,
    'output_dir': '/content/kadastra_outputs/',
}

audit = []

def log(step, rows_before, rows_after, detail=''):
    """Record every row-dropping operation for the audit trail."""
    dropped = rows_before - rows_after
    pct = dropped / rows_before * 100 if rows_before else 0
    audit.append(dict(step=step, rows_before=rows_before, rows_after=rows_after,
                      dropped=dropped, dropped_pct=round(pct,2), detail=detail))
    print(f'  [{step}] {rows_before:,} -> {rows_after:,}  (dropped {dropped:,} = {pct:.1f}%)  {detail}')

os.makedirs(CONFIG['output_dir'], exist_ok=True)
print('Config ready. Outputs:', CONFIG['output_dir'])

Config ready. Outputs: /content/kadastra_outputs/


## Cell 2 — Upload & Load Dataset

The raw file uses a semicolon separator (Tayara exports use `;` rather than `,`). We probe all three common separators and accept the first one that produces more than 3 columns, making the loader robust to minor format changes between scraping runs.

**Expected output:** 30,186 rows × 24 columns.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError('No file uploaded.')

filename  = list(uploaded.keys())[0]
raw_bytes = uploaded[filename]
print(f'File received: {filename}  ({len(raw_bytes):,} bytes)')

df_raw = None
for sep in [';', ',', '\t']:
    try:
        candidate = pd.read_csv(io.BytesIO(raw_bytes), sep=sep, low_memory=False)
        if candidate.shape[1] > 3:
            df_raw = candidate
            print(f'Separator detected: {repr(sep)}')
            break
    except Exception:
        continue

if df_raw is None:
    raise ValueError('Could not parse file — check separator or encoding.')

df_raw.columns = df_raw.columns.str.strip().str.lower()
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols')
print(f'Columns: {list(df_raw.columns)}')

df = df_raw.copy()
TOTAL_START = len(df)
print(f'\nLoaded: {TOTAL_START:,} rows')

Saving final_dfV2-2.csv to final_dfV2-2.csv
File received: final_dfV2-2.csv  (22,470,244 bytes)
Separator detected: ';'
Shape: 30,186 rows x 24 cols
Columns: ['titre', 'adresse', 'localisation', 'description', 'pieces', 'chambres', 'sallesdebain', 'type', 'datepost', 'price_numeric', 'surface_numeric', 'similarity_score', 'link_mubawab', 'link_tayara', 'source', 'meuble', 'neuf', 'parking', 'ascenseur', 'balcon_terrasse', 'climatisation', 'chauffage', 'jardin', 'piscine']

Loaded: 30,186 rows


## Step 1 — Transaction & Category Normalisation

### Problem
The two sources encode property type in incompatible formats:
- **Mubawab:** free-text strings like `"Appartement a vendre"`, `"Maison a louer"`
- **Tayara:** URL slugs like `"appartements"`, `"terrains-et-fermes"`

### Solution
We apply regex patterns against **both** the `type` column and the `titre` column simultaneously. The `titre` field is essential for Tayara rows where `type` is a URL slug that omits the transaction direction entirely.

When a row matches both vente and location patterns (rare but possible for copy-paste listings), we resolve the conflict by trusting the `type` column as the ground truth.

### Output
Two new columns:
- `transaction`: `vente` | `location` | `unknown`
- `category`: `appartement` | `maison` | `terrain` | `local` | `autre` | `unknown`

In [ ]:
print('STEP 1: Type Normalisation')
t     = df['type'].fillna('').str.lower()
titre = df['titre'].fillna('').str.lower()

vente_pat = (
    t.str.contains(r'vend|vente|sale|a vendre', regex=True) |
    titre.str.contains(r'\bvend|\bvente|\bvendre', regex=True)
)
loc_pat = (
    t.str.contains(r'lou|louer|locat|rent|location', regex=True) |
    t.str.contains(r'locations-de-vacances|colocations', regex=True) |
    titre.str.contains(r'\bloue|\blouer|\ba louer', regex=True)
)

df['transaction'] = 'unknown'
df.loc[vente_pat, 'transaction'] = 'vente'
df.loc[loc_pat,   'transaction'] = 'location'

both = vente_pat & loc_pat
df.loc[both, 'transaction'] = df.loc[both, 'type'].str.lower().apply(
    lambda x: 'vente' if 'vend' in str(x) else 'location'
)

cat_patterns = [
    ('appartement', r'appart|appartements|studio|s\d+'),
    ('maison',      r'maison|villa|villas|maisons-et-villas|duplex|triplex|penthouse'),
    ('terrain',     r'terrain|terrains|foncier|ferme|terrains-et-fermes|agricole'),
    ('local',       r'local|bureau|bureaux|commerc|magazin|plateaux|industriel|magasins'),
    ('autre',       r'autre|immobilier|neuf|immeuble|colocation'),
]

df['category'] = 'unknown'
combined_text = t + ' ' + titre
for cat_name, pattern in cat_patterns:
    mask = combined_text.str.contains(pattern, regex=True)
    df.loc[mask & (df['category'] == 'unknown'), 'category'] = cat_name

print(f'  transaction: {df["transaction"].value_counts().to_dict()}')
print(f'  category:    {df["category"].value_counts().to_dict()}')
print('Step 1 complete')

STEP 1: Type Normalisation
  transaction: {'vente': 14763, 'location': 11257, 'unknown': 4166}
  category:    {'appartement': 16572, 'maison': 5492, 'terrain': 4730, 'local': 3112, 'autre': 202, 'unknown': 78}
Step 1 complete


## Step 1b — Immeuble (Whole-Building) Detection

### Problem
The `appartement` category contains a mix of single units and whole buildings (immeubles). Training a price model on this mixture would corrupt it — the price of a whole building is not comparable to a single unit, even after surface normalisation.

### Detection logic
A listing is flagged as an immeuble if it meets **either** of:
1. **Text signal** — unambiguous phrases like `"immeuble de rapport"`, `"plusieurs appartements"`, `"traversant"` (indicates a through-building sold as a block). Note: bare `\bimmeuble\b` is deliberately excluded — in French it can simply mean "the building" and appears in normal single-unit listings ("dans un immeuble bien entretenu").
2. **Structural + price signal** — simultaneously: (rooms > 12 OR surface > 500m²) AND (price > 2M DT for vente), and NOT in a villa context (which could legitimately have large surfaces).

### Post-correction
Any row flagged as immeuble but containing villa/S4-S9 signals is moved to `maison` rather than dropped — these are large villas misdetected as buildings.

### Outcome
Only ~46 immeubles detected. Below the 200-row training threshold, so they are isolated into `predict_pool` for future use when more data arrives.

In [ ]:
print('STEP 1b: Immeuble Detection')
combined = (df['titre'].fillna('') + ' ' + df['description'].fillna('')).str.lower()

TEXT_PATTERN = (
    r'immeuble\s+(?:de\s+rapport|traversant|entier|complet|a\s+vendre|en\s+vente)'
    r'|vente\s+immeuble'
    r'|cession\s+immeuble'
    r'|location\s+immeuble'
    r'|\btraversant\b'
    r'|plusieurs\s+appartements'
    r'|plusieurs\s+logements'
)
text_signal = combined.str.contains(TEXT_PATTERN, regex=True)

pieces_num   = pd.to_numeric(df['pieces'],          errors='coerce')
chambres_num = pd.to_numeric(df['chambres'],        errors='coerce')
surface_num  = pd.to_numeric(df['surface_numeric'], errors='coerce')

structural_signal = (
    (pieces_num   > CONFIG['immeuble_pieces_min'])   |
    (chambres_num > CONFIG['immeuble_chambres_min']) |
    (surface_num  > CONFIG['immeuble_surface_min'])
)

price_num    = pd.to_numeric(df['price_numeric'], errors='coerce')
price_signal = (
    ((df['transaction'] == 'vente')    & (price_num > CONFIG['immeuble_price_vente']))    |
    ((df['transaction'] == 'location') & (price_num > CONFIG['immeuble_price_location']))
)

villa_context = combined.str.contains(
    r'\bvilla\b|\bmaison\b|\bs\+?[456789]\b',
    regex=True
)

is_immeuble = text_signal | (structural_signal & price_signal & ~villa_context)

appt_mask = df['category'] == 'appartement'
df.loc[appt_mask & is_immeuble, 'category'] = 'immeuble'

immeuble_mask = df['category'] == 'immeuble'
villa_signals = combined.str.contains(
    r'\bvilla\b|\bmaison\b|\bs\+?[456789]\b',
    regex=True
)
df.loc[immeuble_mask & villa_signals, 'category'] = 'maison'

n_imm  = (df['category'] == 'immeuble').sum()
n_appt = (df['category'] == 'appartement').sum()
n_mais = (df['category'] == 'maison').sum()

print(f'  Text signal fired (appt rows):       {(appt_mask & text_signal).sum():,}')
print(f'  Structural+price fired (appt rows):  {(appt_mask & structural_signal & price_signal).sum():,}')
print(f'  Final reclassified as immeuble:      {n_imm:,}')
print(f'  Single-unit appartements remaining:  {n_appt:,}')
print(f'  Maison (incl. rescued villa rows):   {n_mais:,}')

sample_cols = [c for c in ['titre','pieces','chambres','surface_numeric','price_numeric','transaction']
               if c in df.columns]
print('\n  Sample detected immeubles:')
for _, row in df[df['category'] == 'immeuble'][sample_cols].head(8).iterrows():
    print(f'    {dict(row)}')
print('Step 1b complete')

STEP 1b: Immeuble Detection
  Text signal fired (appt rows):       38
  Structural+price fired (appt rows):  118
  Final reclassified as immeuble:      46
  Single-unit appartements remaining:  16,522
  Maison (incl. rescued villa rows):   5,496

  Sample detected immeubles:
    {'titre': 'exclusivit foncire immeuble traversant', 'pieces': 3.0, 'chambres': '3.0', 'surface_numeric': 1005.0, 'price_numeric': 6300000.0, 'transaction': 'vente'}
    {'titre': 'immeuble a vendre cite rafaha', 'pieces': 5.0, 'chambres': '5.0', 'surface_numeric': 220.0, 'price_numeric': nan, 'transaction': 'vente'}
    {'titre': 'a vendre  immeuble r8 zone touristique', 'pieces': 8.0, 'chambres': '20.0', 'surface_numeric': 5500.0, 'price_numeric': 13000000.0, 'transaction': 'vente'}
    {'titre': 'immeuble vita  vendre  mjez beb', 'pieces': 11.0, 'chambres': '19.0', 'surface_numeric': 1000.0, 'price_numeric': 3600000.0, 'transaction': 'vente'}
    {'titre': 's3 de 19384 m vue panoramique lac ii', 'pieces': 4.0

## Step 2 — S+N Notation & Floor Extraction

### Problem
52% of rows have a null `pieces` field. However, Tunisian listings systematically use the S+N convention in the `titre` field:
- S0 = studio = 1 pièce
- S1 = 2 pièces
- S2 = 3 pièces, and so on

This convention allows us to recover room count for thousands of otherwise-null rows without imputation.

### Floor extraction
Floor number is a meaningful price predictor (RDC vs 5ème étage can differ by 10-15% in Tunisian markets). We extract it from free text using a combination of:
- A lookup table for common abbreviations (RDC, RDJ)
- Regex for ordinal patterns (`3ème étage`, `au 4`)

### neuf flag enrichment
The structured `neuf` column is sparse. We augment it with text signals (`promoteur`, `livraison`, `nouvellement`) that reliably indicate a new-build property.

In [ ]:
print('STEP 2: S+N and floor extraction')
combined = (df['titre'].fillna('') + ' ' + df['description'].fillna('')).str.lower()

sn_match    = combined.str.extract(r'\bs\+?(\d)\b', expand=False)
df['sn_level'] = pd.to_numeric(sn_match, errors='coerce')

pieces_col     = pd.to_numeric(df['pieces'], errors='coerce')
pieces_from_sn = df['sn_level'].map(CONFIG['sn_to_pieces'])
filled_count   = (pieces_col.isna() & pieces_from_sn.notna()).sum()
df['pieces']   = pieces_col.fillna(pieces_from_sn)
print(f'  Pieces recovered from S+N: {filled_count:,}')

floor_map = {
    'rdj': -1, 'rez de jardin': -1, 'rez-de-jardin': -1,
    'rdc': 0,  'rez de chausse': 0, 'rez-de-chaussee': 0,
}

def extract_floor(text):
    for key, val in floor_map.items():
        if key in text:
            return val
    m = re.search(r'\b(\d{1,2})\s*(?:er|ere|eme|e)\s+(?:etage|tage)', text)
    if m:
        return int(m.group(1))
    m2 = re.search(r'\bau\s+(\d{1,2})\b', text)
    if m2:
        v = int(m2.group(1))
        return v if v <= 15 else None
    return None

df['floor_number'] = combined.apply(extract_floor)
print(f'  Floor number recovered: {df["floor_number"].notna().sum():,}')

if 'neuf' in df.columns:
    neuf_col      = pd.to_numeric(df['neuf'], errors='coerce').fillna(0).astype(int)
    neuf_from_txt = combined.str.contains(r'\bneuf\b|\bnouveau\b|\bpromoteur\b|\blivraison\b', regex=True).astype(int)
    df['neuf']    = ((neuf_col == 1) | (neuf_from_txt == 1)).astype(int)
    print(f'  neuf flag: {df["neuf"].sum():,} listings identified as new-build')

print('Step 2 complete')

STEP 2: S+N and floor extraction
  Pieces recovered from S+N: 5,028
  Floor number recovered: 5,725
  neuf flag: 2,330 listings identified as new-build
Step 2 complete


## Step 3 — Price Cleaning & Log Transform

### Problems identified in EDA
1. **13.3% null prices** (`"Prix à consulter"`) — these listings cannot be used for training. They are moved to `predict_pool.csv` for future inference once a model is trained.
2. **Prices stored in millimes** — some Tayara listings stored prices as millimes (1 DT = 1000 millimes). A 350,000 DT apartment appeared as 350,000,000. We correct these by dividing by 1000 when the result falls within a valid range.
3. **Out-of-range prices** — global floors and ceilings applied per transaction type.
4. **Appartement vente anomalies** — sale prices below 30,000 DT for an apartment are almost certainly rentals or placeholder prices posted in the wrong category. A hard floor of 30,000 DT is applied specifically to `appartement vente` rows.

### Log transform
The target variable `log_price = log1p(price_numeric)` is used rather than raw price for two reasons:
- Price distributions in real estate are log-normal: a symmetric distribution in log space
- Without the transform, the model minimises absolute DT error, meaning a 50,000 DT miss on a 100,000 DT studio is penalised equally to a 50,000 DT miss on a 5,000,000 DT villa. The log transform makes errors proportional across the full price range

To convert predictions back: `price = expm1(log_price_prediction)`

In [ ]:
print('STEP 3: Price Cleaning')
n0 = len(df)
df['price_numeric'] = pd.to_numeric(df['price_numeric'], errors='coerce')

predict_mask = df['price_numeric'].isna()
predict_pool = df[predict_mask].copy()
df = df[~predict_mask].copy()
log('price: null -> predict pool', n0, len(df), f'{predict_mask.sum():,} rows saved for future inference')

vente_mask = df['transaction'] == 'vente'
high_price  = df['price_numeric'] > CONFIG['sale_price_max']
corrected, dropped_millimes = 0, 0

for idx in df[vente_mask & high_price].index:
    val = df.at[idx, 'price_numeric'] / 1000
    if CONFIG['sale_price_min'] <= val <= CONFIG['sale_price_max']:
        df.at[idx, 'price_numeric'] = val
        corrected += 1
    else:
        df.at[idx, 'price_numeric'] = np.nan
        dropped_millimes += 1

print(f'  Millimes correction: {corrected:,} corrected, {dropped_millimes:,} still invalid after correction')
before = len(df)
df = df[df['price_numeric'].notna()].copy()
log('price: uncorrectable high values', before, len(df))

before = len(df)
vente_ok = (df['transaction'] == 'vente') & df['price_numeric'].between(CONFIG['sale_price_min'], CONFIG['sale_price_max'])
loc_ok   = (df['transaction'] == 'location') & df['price_numeric'].between(CONFIG['rent_price_min'], CONFIG['rent_price_max'])
other_ok = ~df['transaction'].isin(['vente', 'location'])
df = df[vente_ok | loc_ok | other_ok].copy()
log('price: out-of-range per transaction type', before, len(df))

before = len(df)
appt_too_cheap = (
    (df['category'] == 'appartement')
    & (df['transaction'] == 'vente')
    & (df['price_numeric'] < 30_000)
)
df = df[~appt_too_cheap].copy()
log('price: appt vente hard floor 30k DT', before, len(df),
    f'dropped {appt_too_cheap.sum()} rentals/placeholders posted as sales')

df['log_price'] = np.log1p(df['price_numeric'])

log_cap = np.log1p(CONFIG['sale_price_max'])
before  = len(df)
df = df[df['log_price'] <= log_cap].copy()
log('log_price: hard cap on residual corrupt values', before, len(df),
    f'cap = {log_cap:.2f} = log1p({CONFIG["sale_price_max"]:,})')

print(f'  log_price range: {df["log_price"].min():.2f} - {df["log_price"].max():.2f}')
print(f'  = {np.expm1(df["log_price"].min()):,.0f} - {np.expm1(df["log_price"].max()):,.0f} DT')
print('Step 3 complete')

STEP 3: Price Cleaning
  [price: null -> predict pool] 30,186 -> 26,171  (dropped 4,015 = 13.3%)  4,015 rows saved for future inference
  Millimes correction: 102 corrected, 1 still invalid after correction
  [price: uncorrectable high values] 26,171 -> 26,170  (dropped 1 = 0.0%)  
  [price: out-of-range per transaction type] 26,170 -> 25,297  (dropped 873 = 3.3%)  
  [price: appt vente hard floor 30k DT] 25,297 -> 25,278  (dropped 19 = 0.1%)  dropped 19 rentals/placeholders posted as sales
  [log_price: hard cap on residual corrupt values] 25,278 -> 25,228  (dropped 50 = 0.2%)  cap = 15.89 = log1p(8,000,000)
  log_price range: 1.10 - 15.89
  = 2 - 8,000,000 DT
Step 3 complete


## Step 4 — Surface Cleaning & Prix/m² Sanity Check

### Problems
- **178 zero-surface rows** — scraping artifact where an empty field was stored as 0 rather than null. Converted to NaN.
- **Category-specific surface caps** — a 600m² appartement is impossible; a 600m² terrain is tiny. Caps are applied per category rather than globally.
- **Prix/m² sanity check (vente only)** — after computing `price/surface`, we drop listings outside [200, 15,000] DT/m². Below 200 DT/m² is physically implausible (cheaper than land in the most remote governorates); above 15,000 DT/m² only exists for a handful of ultra-luxury penthouses in Lac 2 that we verified manually and kept.

In [ ]:
print('STEP 4: Surface Cleaning')
df['surface_numeric'] = pd.to_numeric(df['surface_numeric'], errors='coerce')

zero_count = (df['surface_numeric'] == 0).sum()
df.loc[df['surface_numeric'] == 0, 'surface_numeric'] = np.nan
print(f'  Zeros -> NaN: {zero_count:,} (scraping artifacts)')

surface_caps = {
    'appartement': CONFIG['appt_surface_max'],
    'immeuble':    CONFIG['immeuble_surface_max'],
    'maison':      CONFIG['maison_surface_max'],
    'terrain':     CONFIG['terrain_surface_max'],
    'local':       CONFIG['local_surface_max'],
}

before    = len(df)
drop_mask = pd.Series(False, index=df.index)
for cat, cap in surface_caps.items():
    cat_mask  = df['category'] == cat
    too_large = df['surface_numeric'] > cap
    too_small = (df['surface_numeric'] < CONFIG['surface_min']) & df['surface_numeric'].notna()
    drop_mask |= cat_mask & (too_large | too_small)
df = df[~drop_mask].copy()
log('surface: out-of-range per category', before, len(df))

vente    = df['transaction'] == 'vente'
has_both = df['price_numeric'].notna() & df['surface_numeric'].notna()
df['prix_m2'] = np.nan
df.loc[vente & has_both, 'prix_m2'] = (
    df.loc[vente & has_both, 'price_numeric'] /
    df.loc[vente & has_both, 'surface_numeric']
)

before    = len(df)
bad_ratio = vente & df['prix_m2'].notna() & ~df['prix_m2'].between(CONFIG['prix_m2_min'], CONFIG['prix_m2_max'])
df = df[~bad_ratio].copy()
log('prix_m2: ratio sanity check', before, len(df),
    f'kept [{CONFIG["prix_m2_min"]:,}-{CONFIG["prix_m2_max"]:,}] DT/m2')
print('Step 4 complete')

STEP 4: Surface Cleaning
  Zeros -> NaN: 123 (scraping artifacts)
  [surface: out-of-range per category] 25,228 -> 24,495  (dropped 733 = 2.9%)  
  [prix_m2: ratio sanity check] 24,495 -> 23,829  (dropped 666 = 2.7%)  kept [200-15,000] DT/m2
Step 4 complete


## Step 5 — Pieces/Chambres Consistency Repair

### Problem
EDA revealed 1,162 rows where `pieces < chambres` — a logical impossibility (the number of rooms must include the bedrooms). This is a scraping artifact where the two fields were swapped during extraction.

Additionally, 2,895 rows have `pieces == chambres`, which implies there are no other rooms (no living room, kitchen, etc.) — also implausible for any listed property.

### Repair priority
1. If `pieces` is null but `sn_level` is known → recompute from S+N mapping
2. If `pieces < chambres` → swap the two fields
3. If `pieces == chambres > 0` → set `pieces = chambres + 1` (minimum 1 non-bedroom room)

All repaired rows are flagged with `pieces_imputed = True` so the model can optionally learn that imputed values are less reliable.

In [ ]:
print('STEP 5: Pieces / Chambres Validation')
df['pieces']   = pd.to_numeric(df['pieces'],   errors='coerce')
df['chambres'] = pd.to_numeric(df['chambres'], errors='coerce')
df['pieces_imputed'] = False

structural = df['category'].isin(['appartement', 'maison'])
repaired_sn, repaired_swap, repaired_plus1 = 0, 0, 0

for idx in df[structural].index:
    p  = df.at[idx, 'pieces']
    c  = df.at[idx, 'chambres']
    sn = df.at[idx, 'sn_level'] if 'sn_level' in df.columns else np.nan
    if pd.isna(p) and not pd.isna(sn):
        df.at[idx, 'pieces'] = CONFIG['sn_to_pieces'].get(int(sn), np.nan)
        df.at[idx, 'pieces_imputed'] = True
        repaired_sn += 1
        p = df.at[idx, 'pieces']
    if pd.isna(p) or pd.isna(c):
        continue
    if p < c:
        df.at[idx, 'pieces'], df.at[idx, 'chambres'] = c, p
        df.at[idx, 'pieces_imputed'] = True
        repaired_swap += 1
    elif p == c > 0:
        df.at[idx, 'pieces'] = p + 1
        df.at[idx, 'pieces_imputed'] = True
        repaired_plus1 += 1

print(f'  Repaired via S+N recovery:   {repaired_sn:,}')
print(f'  Repaired via field swap:     {repaired_swap:,}')
print(f'  Repaired via +1 adjustment:  {repaired_plus1:,}')
print('Step 5 complete')

STEP 5: Pieces / Chambres Validation
  Repaired via S+N recovery:   6
  Repaired via field swap:     1,062
  Repaired via +1 adjustment:  2,894
Step 5 complete


## Step 6 — Null Flags (No Imputation)

### Design decision: why not impute here?

Global imputation at preprocessing time would cause **data leakage** in cross-validation. If we fill null `chambres` with the global median, the validation fold's imputed values are computed using information from the training fold — the model sees slightly inflated performance.

Instead, we:
1. Create a `{column}_was_null` binary flag for each numeric feature — this gives the model a way to learn that imputed values are less reliable
2. Leave NaN values as-is — LightGBM handles them natively; Ridge imputation is done with training-fold medians in the modelling notebook

For binary amenity flags (`parking`, `ascenseur`, etc.), Tunisian listings follow a convention of silence = absence: if a listing doesn't mention parking, it almost certainly doesn't have it. So `fillna(0)` is semantically correct for these columns, not imputation.

In [ ]:
print('STEP 6: Create Null Flags (no imputation)')
numeric_features = ['pieces', 'chambres', 'sallesdebain', 'surface_numeric']
binary_flags     = ['meuble','neuf','parking','ascenseur','balcon_terrasse',
                    'climatisation','chauffage','jardin','piscine']

for col in numeric_features:
    if col not in df.columns:
        continue
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[f'{col}_was_null'] = df[col].isna().astype(int)
    print(f'  {col}: {df[col].isna().sum():,} nulls flagged')

for col in binary_flags:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
print(f'  Binary amenity flags ({len(binary_flags)}): null -> 0 (silence = absence convention)')
print('Step 6 complete')

STEP 6: Create Null Flags (no imputation)
  pieces: 7,208 nulls flagged
  chambres: 4,394 nulls flagged
  sallesdebain: 3,691 nulls flagged
  surface_numeric: 2,127 nulls flagged
  Binary amenity flags (9): null -> 0 (silence = absence convention)
Step 6 complete


## Step 7 — Feature Engineering

This is the most important step. We create three families of features:

### 7a. Spatial features
GPS coordinates are parsed from the `localisation` column and validated against Tunisia's bounding box. Haversine distances are computed to 7 reference points:
- **Tunis centre, Lac, La Marsa** — the three Tunis price axes
- **Hammamet, Sousse, Sfax, Djerba** — major regional markets
- **dist_to_coast** — minimum of 5 coastal reference distances

For rows without GPS coordinates (~27%), we impute using the **governorate median lat/lon** — far better than a global median, since a listing in Sfax should not get Tunis coordinates.

### 7b. Zone extraction
Tunisian addresses follow the pattern `"Cité X À [Zone]"`. We extract the zone after the `À` separator. This gives us 157 distinct zones — a much finer spatial grain than the 18 governorates, enabling zone-level price medians in the modelling notebook.

### 7c. Temporal features
Listing age, year, month, and day-of-week capture seasonal patterns and market drift across the 510-day scraping window (Oct 2024 – Mar 2026).

### 7d. NLP flags
19 binary flags extracted from `titre + description` using domain-specific regex patterns. Key signals:
- `haut_standing` — price premium of 20-40%
- `vue_mer` — sea view, significant coastal premium
- `titre_bleu` — individual land title, reduces legal risk premium
- `constructible` — critical for terrain valuation

In [ ]:
print('STEP 7: Feature Engineering')

import math
import unicodedata
from difflib import get_close_matches

def strip_accents(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )


def haversine(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in km between two (lat, lon) points."""
    R = 6371.0
    lat1_r = np.radians(lat1)
    lon1_r = np.radians(lon1)
    lat2_r = np.radians(lat2)
    lon2_r = np.radians(lon2)
    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r
    a = np.sin(dlat/2)**2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

REF_POINTS = {
    'tunis_centre': (36.8190, 10.1658),
    'lac':          (36.8320, 10.2290),
    'la_marsa':     (36.8781, 10.3247),
    'hammamet':     (36.3996, 10.5491),
    'sousse':       (35.8256, 10.6369),
    'sfax':         (34.7406, 10.7603),
    'djerba':       (33.8764, 10.8474),
    'coast_bizerte': (37.2744, 9.8739),
    'coast_nabeul':  (36.4511, 10.7357),
    'coast_monastir':(35.7643, 10.8113),
    'coast_sfax':    (34.7406, 10.7603),
    'coast_gabes':   (33.8831, 10.0982),
}

if 'localisation' in df.columns:
    coords_extracted = df['localisation'].str.extract(
        r'([-\d.]+)\s*,\s*([-\d.]+)', expand=True
    )
    df['lat'] = pd.to_numeric(coords_extracted[0], errors='coerce')
    df['lon'] = pd.to_numeric(coords_extracted[1], errors='coerce')
    bad_lat = ~df['lat'].between(30.2, 37.5)
    bad_lon = ~df['lon'].between(7.5, 11.6)
    df.loc[bad_lat | bad_lon, ['lat', 'lon']] = np.nan
    n_coords = df['lat'].notna().sum()
    df['has_coords'] = df['lat'].notna().astype(int)
    print(f'  Coordinates parsed: {n_coords:,}/{len(df):,} ({n_coords/len(df)*100:.1f}%) valid')
else:
    df['lat'] = np.nan
    df['lon'] = np.nan
    df['has_coords'] = 0
    print('  No localisation column found — coordinates skipped')

has_c = df['lat'].notna()
for ref_name, (ref_lat, ref_lon) in REF_POINTS.items():
    col = f'dist_{ref_name}'
    df[col] = np.nan
    df.loc[has_c, col] = haversine(
        df.loc[has_c, 'lat'].values, df.loc[has_c, 'lon'].values,
        ref_lat, ref_lon
    )
    fill_val = df.loc[has_c, col].median()
    df[col] = df[col].fillna(fill_val if pd.notna(fill_val) else 50.0)

coast_cols = [c for c in df.columns if c.startswith('dist_coast_')]
if coast_cols:
    df['dist_to_coast'] = df[coast_cols].min(axis=1)
    print(f'  dist_to_coast computed from {len(coast_cols)} coastal reference points')

for col in ['dist_tunis_centre', 'dist_to_coast', 'dist_lac', 'dist_la_marsa']:
    if col in df.columns:
        print(f'  {col}: {df[col].min():.1f} - {df[col].max():.1f} km (median {df[col].median():.1f} km)')

if 'datepost' in df.columns:
    df['datepost'] = pd.to_datetime(df['datepost'], dayfirst=True, errors='coerce')
    ref = df['datepost'].max()
    df['listing_age_days'] = (ref - df['datepost']).dt.days
    df['listing_age_days'] = df['listing_age_days'].fillna(df['listing_age_days'].median())
    df['year_posted']        = df['datepost'].dt.year
    df['month_posted']       = df['datepost'].dt.month
    df['day_of_week_posted'] = df['datepost'].dt.dayofweek
    print(f'  listing_age_days: 0 - {df["listing_age_days"].max():.0f} days')

if 'adresse' in df.columns:
    df['zone'] = (df['adresse']
                  .str.extract(r'[A\u00c0]\s+(.+)$', expand=False)
                  .str.strip())
    df['zone'] = df['zone'].fillna(df['adresse'].str.split().str[0])
elif 'localisation' in df.columns:
    df['zone'] = (df['lat'].round(2).astype(str) + '_' + df['lon'].round(2).astype(str))

print(f'  Zone extracted: {df["zone"].nunique()} unique zones')

if 'adresse' in df.columns:
    df['sub_zone'] = (df['adresse']
                      .str.extract(r'^(.+?)\s+[\u00e0\u00c0a]\s+', expand=False)
                      .str.strip())
    df['sub_zone'] = (df['sub_zone'].str.lower()
                      .str.replace(r'^cit\u00e9\s+', '', regex=True))
    df['sub_zone'] = df['sub_zone'].fillna(df['adresse'].str.split().str[0])
    print(f'  Sub-zone extracted: {df["sub_zone"].nunique()} unique sub-zones')

# ================================================================
# Cell 10 — Improved governorate extraction with fuzzy fallback
# ================================================================
from difflib import get_close_matches

# Reference: all known Tunisian delegations mapped to their governorate
# This list is fixed geography — it doesn't change with scraping runs
DELEGATION_TO_GOV = {
    # Tunis
    'tunis': 'Tunis', 'bab bhar': 'Tunis', 'bab souika': 'Tunis',
    'carthage': 'Tunis', 'el menzah': 'Tunis', 'el manar': 'Tunis',
    'le bardo': 'Tunis', 'den den': 'Tunis', 'douar hicher': 'Tunis',
    'sijoumi': 'Tunis', 'ettadhamen': 'Tunis', 'ezzouhour': 'Tunis',
    'la medina': 'Tunis', 'cite el khadra': 'Tunis',
    'sidi hassine': 'Tunis',      # delegation in Tunis
    'el omrane': 'Tunis',         # delegation in Tunis
    'el omrane superieur': 'Tunis',
    'cite jaouhara': 'Tunis',     # residential area in Tunis
    'el maamoura': 'Nabeul',      # coastal area near Nabeul
    # Ariana
    'ariana': 'Ariana', 'la soukra': 'Ariana', 'raoued': 'Ariana',
    'kalaat landlous': 'Ariana', 'sidi thabet': 'Ariana',
    'mnihla': 'Ariana', 'ghazela': 'Ariana', 'ennasr': 'Ariana',
    # Ben Arous
    'ben arous': 'Ben Arous', 'rades': 'Ben Arous', 'hammam lif': 'Ben Arous',
    'bou mhel': 'Ben Arous', 'megrine': 'Ben Arous', 'fouchana': 'Ben Arous',
    'mornag': 'Ben Arous', 'hammam chott': 'Ben Arous', 'el mourouj': 'Ben Arous',
    'nouvelle medina': 'Ben Arous', 'el yasminette': 'Ben Arous',
    # Manouba
    'manouba': 'Manouba', 'oued ellil': 'Manouba', 'tebourba': 'Manouba',
    'jedaida': 'Manouba', 'douar hicher': 'Manouba', 'borj el amri': 'Manouba',
    # La Marsa / Carthage coast
    'la marsa': 'La Marsa', 'gammarth': 'La Marsa', 'sidi bou said': 'La Marsa',
    'aouina': 'La Marsa', 'le kram': 'La Marsa', 'la goulette': 'La Marsa',
    'les berges du lac': 'Lac', 'lac 2': 'Lac', 'lac 1': 'Lac',
    # Bizerte
    'bizerte': 'Bizerte', 'menzel bourguiba': 'Bizerte', 'mateur': 'Bizerte',
    'menzel jemil': 'Bizerte', 'sejnane': 'Bizerte', 'ras jebel': 'Bizerte',
    'zarzouna': 'Bizerte', 'tinja': 'Bizerte',
    'raf raf': 'Bizerte',         # coastal village in Bizerte
    'ghar el melh': 'Bizerte',    # coastal village in Bizerte
    'barraket essahel': 'Ben Arous', # coastal area south of Tunis
    # Nabeul
    'nabeul': 'Nabeul', 'hammamet': 'Nabeul', 'kelibia': 'Nabeul',
    'korba': 'Nabeul', 'beni khalled': 'Nabeul', 'dar chaabane': 'Nabeul',
    'soliman': 'Nabeul', 'grombalia': 'Nabeul', 'menzel temime': 'Nabeul',
    'bou argoub': 'Nabeul', 'takelsa': 'Nabeul', 'somaa': 'Nabeul',
    'mrezga': 'Nabeul', 'el haouaria': 'Nabeul', 'haouaria': 'Nabeul',
    'beni khiar': 'Nabeul',       # delegation in Nabeul
    'tantana': 'Nabeul',
    # Zaghouan
    'zaghouan': 'Zaghouan', 'el fahs': 'Zaghouan', 'zriba': 'Zaghouan',
    'bir mcherga': 'Zaghouan', 'nadhour': 'Zaghouan',
    # Sousse
    'sousse': 'Sousse', 'sahloul': 'Sousse', 'kantaoui': 'Sousse',
    'msaken': 'Sousse', 'akouda': 'Sousse', 'kalaa kebira': 'Sousse',
    'enfidha': 'Sousse', 'hergla': 'Sousse', 'sidi bou ali': 'Sousse',
    'kondar': 'Sousse', 'kalaa sghira': 'Sousse',
    'khezama': 'Sousse',          # Khezama Est / Ouest — Sousse suburb
    'khezama est': 'Sousse',
    'khezama ouest': 'Sousse',
    'kantaoui': 'Sousse',
    'hammam el ghezaz': 'Nabeul', # coastal village in Nabeul governorate
    'hammam ghezeze': 'Nabeul',   # same place, variant spelling
    'salakta': 'Mahdia',          # coastal town in Mahdia
    'rejiche': 'Mahdia',          # delegation in Mahdia
    'ezzahra': 'Ben Arous',       # suburb of Ben Arous

    # Monastir
    'monastir': 'Monastir', 'skanes': 'Monastir', 'ksar hellal': 'Monastir',
    'moknine': 'Monastir', 'jemmal': 'Monastir', 'ksibet': 'Monastir',
    'bembla': 'Monastir', 'sayada': 'Monastir', 'teboulba': 'Monastir',
    'bekalta': 'Monastir', 'sahline': 'Monastir',
    # Mahdia
    'mahdia': 'Mahdia', 'ksour essef': 'Mahdia', 'chebba': 'Mahdia',
    'el jem': 'Mahdia', 'bou merdes': 'Mahdia', 'sidi alouane': 'Mahdia',
    # Sfax
    'sfax': 'Sfax', 'sakiet ezzit': 'Sfax', 'sakiet eddaier': 'Sfax',
    'thyna': 'Sfax', 'agareb': 'Sfax', 'el hencha': 'Sfax',
    'menzel chaker': 'Sfax', 'bir ali ben khalifa': 'Sfax',
    # Kairouan
    'kairouan': 'Kairouan', 'sbikha': 'Kairouan', 'haffouz': 'Kairouan',
    'oueslatia': 'Kairouan', 'el alaa': 'Kairouan', 'nasrallah': 'Kairouan',
    'alqayrawan': 'Kairouan', 'bou arada': 'Siliana',
    # Siliana
    'siliana': 'Siliana', 'bou arada': 'Siliana', 'makthar': 'Siliana',
    'el krib': 'Siliana', 'rouhia': 'Siliana', 'kesra': 'Siliana',
    # Kasserine
    'kasserine': 'Kasserine', 'sbeitla': 'Kasserine', 'feriana': 'Kasserine',
    'thala': 'Kasserine', 'foussana': 'Kasserine',
    # Kef
    'le kef': 'Kef', 'tajerouine': 'Kef', 'dahmani': 'Kef',
    'nebeur': 'Kef', 'sakiet sidi youssef': 'Kef',
    # Jendouba
    'jendouba': 'Jendouba', 'tabarka': 'Jendouba', 'ain draham': 'Jendouba',
    'beja': 'Beja', 'nefza': 'Jendouba', 'fernana': 'Jendouba',
    # Beja
    'beja': 'Beja', 'testour': 'Beja', 'mjez el bab': 'Beja',
    'teboursouk': 'Beja', 'nefza': 'Beja', 'amdoun': 'Beja',
    # Gabes
    'gabes': 'Gabes', 'mareth': 'Gabes', 'matmata': 'Gabes',
    'el hamma': 'Gabes', 'nouvelle matmata': 'Gabes',
    'gabès': 'Gabes',
    # Gafsa
    'gafsa': 'Gafsa', 'metlaoui': 'Gafsa', 'redeyef': 'Gafsa',
    'moulares': 'Gafsa', 'sened': 'Gafsa', 'el ksar': 'Gafsa',
    # Tozeur
    'tozeur': 'Tozeur', 'nefta': 'Tozeur', 'degache': 'Tozeur',
    # Kebili
    'kebili': 'Kebili', 'douz': 'Kebili', 'souk lahad': 'Kebili',
    # Medenine / Djerba
    'medenine': 'Medenine', 'djerba': 'Medenine', 'zarzis': 'Medenine',
    'midoun': 'Medenine', 'houmt souk': 'Medenine', 'ben gardane': 'Medenine',
    'aghir': 'Medenine',          # village on Djerba island
    'tezdaine': 'Medenine',       # village on Djerba island
    # Tataouine
    'tataouine': 'Tataouine', 'remada': 'Tataouine', 'ghomrassen': 'Tataouine',
}

# Normalise dictionary keys for fast lookup
NORMALISED_DELEGATION_MAP = {strip_accents(k): v for k, v in DELEGATION_TO_GOV.items()}
ALL_LOCALITIES = sorted(NORMALISED_DELEGATION_MAP.keys())

def extract_gouvernorat(adresse):
    if pd.isna(adresse):
        return 'unknown'
    s = strip_accents(str(adresse).lower().strip())
    # Extract part after "à" if present
    match = re.search(r'\bà\s+(.+)$', s)
    zone = match.group(1).strip() if match else s

    # 1. Direct lookup
    if zone in NORMALISED_DELEGATION_MAP:
        return NORMALISED_DELEGATION_MAP[zone]
    if s in NORMALISED_DELEGATION_MAP:
        return NORMALISED_DELEGATION_MAP[s]

    # 2. Substring match (longest first)
    for key in sorted(NORMALISED_DELEGATION_MAP.keys(), key=len, reverse=True):
        if key in s:
            return NORMALISED_DELEGATION_MAP[key]

    # 3. Fuzzy match on zone
    close = get_close_matches(zone, ALL_LOCALITIES, n=1, cutoff=0.82)
    if close:
        return NORMALISED_DELEGATION_MAP[close[0]]

    # 4. Fuzzy match on full address
    close = get_close_matches(s, ALL_LOCALITIES, n=1, cutoff=0.82)
    if close:
        return NORMALISED_DELEGATION_MAP[close[0]]

    return 'unknown'



if 'adresse' in df.columns:
    df['gouvernorat'] = df['adresse'].apply(extract_gouvernorat)

    if has_c.any():
        gov_lat_med = df.loc[has_c].groupby('gouvernorat')['lat'].median()
        gov_lon_med = df.loc[has_c].groupby('gouvernorat')['lon'].median()
        missing_c   = df['lat'].isna()
        df.loc[missing_c, 'lat'] = df.loc[missing_c, 'gouvernorat'].map(gov_lat_med)
        df.loc[missing_c, 'lon'] = df.loc[missing_c, 'gouvernorat'].map(gov_lon_med)

        newly_filled = missing_c & df['lat'].notna()
        if newly_filled.any():
            for ref_name, (ref_lat, ref_lon) in REF_POINTS.items():
                col = f'dist_{ref_name}'
                df.loc[newly_filled, col] = haversine(
                    df.loc[newly_filled, 'lat'].values,
                    df.loc[newly_filled, 'lon'].values,
                    ref_lat, ref_lon
                )
            if coast_cols:
                df['dist_to_coast'] = df[coast_cols].min(axis=1)
            print(f'  Coordinates imputed via governorate median: {newly_filled.sum():,} rows recovered')

    gov_categories = sorted(df['gouvernorat'].unique())
    gov_to_int     = {g: i for i, g in enumerate(gov_categories)}
    df['gouvernorat_cat'] = df['gouvernorat'].map(gov_to_int).astype(int)
    n_mapped = (df['gouvernorat'] != 'unknown').sum()
    print(f'  Governorate coverage: {n_mapped:,}/{len(df):,} ({n_mapped/len(df)*100:.1f}%)')

if 'surface_numeric' in df.columns:
    df['log_surface'] = np.log1p(pd.to_numeric(df['surface_numeric'], errors='coerce'))

if 'pieces' in df.columns and 'surface_numeric' in df.columns:
    df['pieces_per_100m2'] = (
        df['pieces'] / df['surface_numeric'].replace(0, np.nan) * 100
    ).round(2).clip(0, 20)

outdoor_cols = [c for c in ['balcon_terrasse','jardin'] if c in df.columns]
if outdoor_cols:
    df['has_outdoor'] = df[outdoor_cols].max(axis=1).astype(int)

premium_cols = [c for c in ['piscine','ascenseur','parking'] if c in df.columns]
if premium_cols:
    df['premium_score'] = df[premium_cols].sum(axis=1)

print('  Extracting NLP flags from title + description text...')
combined_text = (df['titre'].fillna('') + ' ' + df['description'].fillna('')).str.lower()

nlp_patterns = [
    ('vue_mer',          r'vue\s+(?:sur\s+)?(?:la\s+)?mer|vue\s+mer|face\s+mer'),
    ('vue_panoramique',  r'vue\s+panor'),
    ('haut_standing',    r'haut\s+standing|grand\s+standing|haut\-standing'),
    ('etat_neuf',        r'\bneuf\b|nouvellement|refait\s+a\s+neuf|etat\s+neuf'),
    ('cuisine_equipee',  r'cuisine\s+(?:equip|am\xe9nag|install)'),
    ('double_vitrage',   r'double\s+vitrage|double\-vitrage'),
    ('gardien',          r'\bgardien\b|gardienne|surveillance\s+24'),
    ('interphone',       r'interphone|visiophone|digicode'),
    ('titre_bleu',       r'titre\s+(?:bleu|foncier|individuel)'),
    ('proche_mer',       r'proche\s+(?:de\s+la\s+)?(?:mer|plage|littor)|deux\s+pas\s+(?:de\s+la\s+)?(?:mer|plage)'),
    ('proche_centre',    r'proche\s+(?:du\s+)?(?:centre|commodit|transport|m\xe9tro)'),
    ('residence_gardee', r'r\xe9sidence\s+(?:gard\xe9e|s\xe9curis\xe9e|ferm\xe9e)'),
    ('agricole',         r'agricole|ferme|champ'),
    ('constructible',    r'constructible|viabilis|terrain\s+[\w\s]*construct'),
    ('boutique',         r'boutique|commerce|magasin|vitrine'),
    ('entrepot',         r'entrep\u00f4t|stockage|d\u00e9p\u00f4t'),
    ('bureau',           r'bureau|plateau|open\s*space'),
    ('rez_de_chaussee',  r'rez\s*[\-\s]*de\s+chauss\u00e9e|rdc'),
    ('dernier_etage',    r'dernier\s+\u00e9tage|\u00e9tage\s+\u00e9lev\u00e9'),
]

for flag_name, pattern in nlp_patterns:
    df[f'nlp_{flag_name}'] = (
        combined_text.str.contains(pattern, regex=True).astype(int)
    )
    cnt = df[f'nlp_{flag_name}'].sum()
    if cnt > 10:
        print(f'    {flag_name:<20}: {cnt:5,} rows ({cnt/len(df)*100:.1f}%)')

print('  NLP flags added.')
print('Step 7 complete')

STEP 7: Feature Engineering
  Coordinates parsed: 17,406/23,829 (73.0%) valid
  dist_to_coast computed from 5 coastal reference points
  dist_tunis_centre: 0.0 - 536.7 km (median 15.6 km)
  dist_to_coast: 0.0 - 215.2 km (median 58.2 km)
  dist_lac: 0.0 - 537.6 km (median 9.9 km)
  dist_la_marsa: 0.1 - 542.1 km (median 11.6 km)
  listing_age_days: 0 - 510 days
  Zone extracted: 157 unique zones
  Sub-zone extracted: 476 unique sub-zones
  Coordinates imputed via governorate median: 6,423 rows recovered
  Governorate coverage: 23,544/23,829 (98.8%)
  Extracting NLP flags from title + description text...
    vue_mer             : 1,052 rows (4.4%)
    vue_panoramique     :   340 rows (1.4%)
    haut_standing       : 2,579 rows (10.8%)
    etat_neuf           : 1,292 rows (5.4%)
    cuisine_equipee     :    80 rows (0.3%)
    double_vitrage      :   440 rows (1.8%)
    gardien             :   317 rows (1.3%)
    interphone          :   159 rows (0.7%)
    titre_bleu          : 1,979 rows (

In [ ]:
# Add after the gouvernorat extraction loop in Cell 10:
n_unknown = (df['gouvernorat'] == 'unknown').sum()
n_total   = len(df)
print(f'Governorate coverage: {n_total - n_unknown:,}/{n_total:,} '
      f'({(n_total - n_unknown)/n_total*100:.1f}%)')
print(f'Unknown remaining: {n_unknown:,}')

# Show what the unknown addresses look like
print('\nSample unknown addresses:')
print(df[df['gouvernorat'] == 'unknown']['adresse'].value_counts().head(20))

Governorate coverage: 23,544/23,829 (98.8%)
Unknown remaining: 285

Sample unknown addresses:
adresse
Borj Cedria À Hammam Chatt       70
Bellevue À El Ouardia            20
Hammam Chatt À Hammam Chatt      13
Cité Hached                      12
El Ouardia À El Ouardia          12
Sidi Abdelhamid                  11
Cité Bou Akroucha À Mohamadia    11
Bouficha                         10
Mezraya                           8
El Hrairia À El Hrairia           8
Mohamadia À Mohamadia             6
Ettahrir À Ettahrir               5
Bir El Bey À Hammam Chatt         5
Utique                            5
El Mida                           4
Ajim À Ajim                       4
Mahrès À Mahrès                   4
Ibn Sina À El Kabaria             4
Metline À Metline                 4
Cité Ettahrir Sup À Ettahrir      3
Name: count, dtype: int64


## Step 7b — Zone-Relative Price Floor

### Problem
Despite the global price floors applied in Step 3, some listings survive with prices that are clearly anomalous relative to their neighbourhood. Specifically:
- **Location listings** priced at 200-250 DT/month in zones where the median rent is 1,500-3,100 DT/month — these are weekly rates, per-room rates, or symbolic placeholder prices
- **Vente listings** priced at 25,000-50,000 DT in zones where the median sale price is 500,000+ DT — likely rentals or placeholders posted in the wrong category

### Solution
Compute **separate zone medians for location and vente** (never mixed — a zone median of 400,000 DT driven by sale prices has no meaning as a floor for a 600 DT/month rental). Drop listings priced below a fraction of their zone's median:
- Location: drop if `price < 35%` of zone median location price
- Vente: drop if `price < 50%` of zone median vente price

The filter only activates for zones with ≥ 10 listings of the same transaction type, preventing it from incorrectly dropping listings in under-represented zones where the median is unstable.

**Result:** 284 suspicious location + 1,594 suspicious vente listings removed.

In [ ]:
print('STEP 7b: Zone-relative price floor')

for _col in ['zone_med_loc','zone_cnt_loc','zone_med_vente','zone_cnt_vente']:
    if _col in df.columns:
        df = df.drop(columns=[_col])

ZONE_FLOOR_LOCATION = 0.35
ZONE_FLOOR_VENTE    = 0.50
ZONE_MIN_COUNT      = 10

if 'zone' not in df.columns:
    print('  Zone column missing - skipping zone-relative price floor')
else:
    before = len(df)

    loc_mask   = df['transaction'] == 'location'
    vente_mask = df['transaction'] == 'vente'

    loc_zone_med = (df[loc_mask].groupby('zone')['price_numeric'].median().rename('zone_med_loc'))
    loc_zone_cnt = (df[loc_mask].groupby('zone').size().rename('zone_cnt_loc'))
    vente_zone_med = (df[vente_mask].groupby('zone')['price_numeric'].median().rename('zone_med_vente'))
    vente_zone_cnt = (df[vente_mask].groupby('zone').size().rename('zone_cnt_vente'))

    df = (df.join(loc_zone_med, on='zone').join(loc_zone_cnt, on='zone')
            .join(vente_zone_med, on='zone').join(vente_zone_cnt, on='zone'))

    suspicious = pd.Series(False, index=df.index)

    loc_susp = (
        loc_mask & df['zone_med_loc'].notna()
        & (df['zone_cnt_loc'] >= ZONE_MIN_COUNT)
        & (df['price_numeric'] < ZONE_FLOOR_LOCATION * df['zone_med_loc'])
    )
    suspicious |= loc_susp
    print(f'  Suspicious location listings: {loc_susp.sum():,}')
    if loc_susp.any():
        cols = [c for c in ['adresse','price_numeric','zone_med_loc','surface_numeric'] if c in df.columns]
        print(df[loc_susp][cols].head(8).to_string())

    vente_susp = (
        vente_mask & df['zone_med_vente'].notna()
        & (df['zone_cnt_vente'] >= ZONE_MIN_COUNT)
        & (df['price_numeric'] < ZONE_FLOOR_VENTE * df['zone_med_vente'])
    )
    suspicious |= vente_susp
    print(f'  Suspicious vente listings:    {vente_susp.sum():,}')
    if vente_susp.any():
        cols = [c for c in ['adresse','price_numeric','zone_med_vente','surface_numeric'] if c in df.columns]
        print(df[vente_susp][cols].head(8).to_string())

    df = df[~suspicious].drop(
        columns=['zone_med_loc', 'zone_cnt_loc', 'zone_med_vente', 'zone_cnt_vente']
    )
    log('price: zone-relative floor', before, len(df),
        f'dropped {loc_susp.sum()} location + {vente_susp.sum()} vente')

print('Step 7b complete')

STEP 7b: Zone-relative price floor
  Suspicious location listings: 284
                          adresse  price_numeric  zone_med_loc  surface_numeric
51        Ain Zaghouan À La Marsa          850.0        3100.0             75.0
378  Ain Zaghouan Nord À La Marsa          700.0        3100.0            127.0
499  Ain Zaghouan Nord À La Marsa          200.0        3100.0             90.0
838  Cité Ennasr 2 À Ariana Ville          500.0        1500.0             18.0
845  Ain Zaghouan Nord À La Marsa         1050.0        3100.0            100.0
855             Aouina À La Marsa          850.0        3100.0             55.0
916             Aouina À La Marsa          900.0        3100.0             60.0
919       Ain Zaghouan À La Marsa         1060.0        3100.0             70.0
  Suspicious vente listings:    1,594
                         adresse  price_numeric  zone_med_vente  surface_numeric
4              Aouina À La Marsa       245000.0        570000.0             90.0
5        

## Step 7c — Terrain Double-Null Removal

### Problem
Diagnostic analysis revealed that 145 terrain rows have null surface, and among these, a subset also have null zone. For a terrain listing:
- Without surface → cannot compute prix/m², the primary terrain price driver
- Without zone → cannot use zone median as a fallback spatial signal

Both conditions together make a listing **unlearnable** — the model can only predict the global mean for it, producing ~21% median APE on this subset vs ~5% for surface-known terrains.

### Decision
Drop only rows that are simultaneously surface-null AND zone-null. Rows with null surface but known zone are **kept** — they can still use zone price median as a signal.

These are typically large agricultural or subdivision plots where the seller deliberately omits surface (sold by negotiation, or surface is variable).

In [ ]:
print('STEP 7c: Terrain double-null removal')
before = len(df)
terrain_mask  = df['category'] == 'terrain'
surf_null     = df['surface_numeric'].isna()
zone_null     = df['zone'].isna() | (df['zone'].str.strip() == '')
terrain_blind = terrain_mask & surf_null & zone_null
df = df[~terrain_blind].copy()
log('terrain: double-null (no surface + no zone)', before, len(df),
    f'dropped {terrain_blind.sum()} unlearnable terrain rows')
print('Step 7c complete')

STEP 7c: Terrain double-null removal
  [terrain: double-null (no surface + no zone)] 21,951 -> 21,951  (dropped 0 = 0.0%)  dropped 0 unlearnable terrain rows
Step 7c complete


## Steps 8 & 9 — Sub-Dataset Split & Feature Manifests

### Why separate models per sub-type?
Price drivers differ fundamentally across property types:
- **Appartement:** surface, floor number, amenities, zone median → 57 features
- **Terrain:** surface + zone (almost nothing else) → 41 features
- **Local commercial:** entirely different dynamics (footfall, commercial zoning)

A single combined model produces a compromise that is suboptimal for all types. Separate models allow each to be tuned to its specific feature set.

### Trainable sub-datasets (≥ 200 rows)
| Sub-dataset | Rows | Notes |
|---|---|---|
| appt_location | 7,250 | Largest — trained first |
| appt_vente | 4,383 | Primary target |
| maison_vente | 2,636 | Heterogeneous (villas + houses) |
| maison_location | 1,105 | |
| terrain_vente | 1,509 | Simple feature set |
| local_vente | 440 | Small — use conservative model |

### Feature manifests
Each sub-dataset's feature manifest is computed dynamically — only features that actually exist in the dataframe are included. This prevents errors when optional features are absent.

In [ ]:
print('STEP 8: Split into sub-datasets')
subsets = {
    'appt_vente':       (df['category'] == 'appartement') & (df['transaction'] == 'vente'),
    'appt_location':    (df['category'] == 'appartement') & (df['transaction'] == 'location'),
    'immeuble_vente':   (df['category'] == 'immeuble')    & (df['transaction'] == 'vente'),
    'immeuble_location':(df['category'] == 'immeuble')    & (df['transaction'] == 'location'),
    'maison_vente':     (df['category'] == 'maison')      & (df['transaction'] == 'vente'),
    'maison_location':  (df['category'] == 'maison')      & (df['transaction'] == 'location'),
    'terrain_vente':    (df['category'] == 'terrain')     & (df['transaction'] == 'vente'),
    'local_vente':      (df['category'] == 'local')       & (df['transaction'] == 'vente'),
}

datasets = {}
for name, mask in subsets.items():
    sub = df[mask].copy()
    n   = len(sub)
    if n >= CONFIG['min_rows_to_train']:
        datasets[name] = sub
        print(f'  {name:<25}: {n:,} rows  -> TRAINABLE')
    else:
        print(f'  {name:<25}: {n:,} rows  -> SKIPPED (below {CONFIG["min_rows_to_train"]} row threshold)')

print('\nSTEP 9: Feature manifests')

def get_feature_set(dataset_name, df_cols):
    structural       = ['surface_numeric', 'log_surface', 'pieces', 'chambres', 'sallesdebain']
    appt_extra       = ['floor_number', 'sn_level', 'has_outdoor', 'ascenseur', 'neuf',
                        'premium_score', 'pieces_per_100m2']
    amenities        = ['parking', 'balcon_terrasse', 'climatisation', 'chauffage', 'jardin', 'piscine']
    spatial_core     = ['gouvernorat_cat', 'has_coords', 'dist_tunis_centre', 'dist_to_coast',
                        'dist_lac', 'dist_la_marsa']
    spatial_extended = spatial_core + ['lat', 'lon', 'dist_hammamet', 'dist_sousse',
                                       'dist_sfax', 'dist_djerba']
    temporal         = ['listing_age_days', 'year_posted', 'month_posted', 'day_of_week_posted']
    null_flags       = [c for c in df_cols if c.endswith('_was_null')]
    nlp_flags        = [c for c in df_cols if c.startswith('nlp_')]

    specs = {
        'appt_vente':     structural + appt_extra + amenities + spatial_extended + temporal + null_flags + nlp_flags,
        'appt_location':  structural + appt_extra + amenities + ['meuble'] + spatial_extended + temporal + null_flags + nlp_flags,
        'maison_vente':   structural + ['has_outdoor', 'neuf', 'premium_score', 'pieces_per_100m2'] + amenities + spatial_extended + temporal + null_flags + nlp_flags,
        'maison_location':structural + ['has_outdoor', 'premium_score'] + amenities + ['meuble'] + spatial_extended + temporal + null_flags + nlp_flags,
        'terrain_vente':  ['surface_numeric', 'log_surface'] + spatial_extended + temporal + null_flags + nlp_flags,
        'local_vente':    structural[:3] + amenities[:3] + spatial_core + temporal + null_flags + nlp_flags,
    }
    feats = specs.get(dataset_name, structural + amenities + spatial_core + temporal + nlp_flags)
    return [f for f in feats if f in df_cols]

FEATURE_MANIFESTS = {}
for name, sub_df in datasets.items():
    feats = get_feature_set(name, list(sub_df.columns))
    FEATURE_MANIFESTS[name] = feats
    print(f'\n  [{name}]  n={len(sub_df):,}  {len(feats)} features')
    print(f'  {feats}')

print('\nSteps 8 & 9 complete')

STEP 8: Split into sub-datasets
  appt_vente               : 4,383 rows  -> TRAINABLE
  appt_location            : 7,250 rows  -> TRAINABLE
  immeuble_vente           : 15 rows  -> SKIPPED (below 200 row threshold)
  immeuble_location        : 3 rows  -> SKIPPED (below 200 row threshold)
  maison_vente             : 2,636 rows  -> TRAINABLE
  maison_location          : 1,105 rows  -> TRAINABLE
  terrain_vente            : 1,509 rows  -> TRAINABLE
  local_vente              : 440 rows  -> TRAINABLE

STEP 9: Feature manifests

  [appt_vente]  n=4,383  57 features
  ['surface_numeric', 'log_surface', 'pieces', 'chambres', 'sallesdebain', 'floor_number', 'sn_level', 'has_outdoor', 'ascenseur', 'neuf', 'premium_score', 'pieces_per_100m2', 'parking', 'balcon_terrasse', 'climatisation', 'chauffage', 'jardin', 'piscine', 'gouvernorat_cat', 'has_coords', 'dist_tunis_centre', 'dist_to_coast', 'dist_lac', 'dist_la_marsa', 'lat', 'lon', 'dist_hammamet', 'dist_sousse', 'dist_sfax', 'dist_djerba', '

## Step 10 — Save & Download Outputs

### Output files produced

For each trainable sub-dataset:
- **`clean_{name}.csv`** — full cleaned dataframe with all columns (for EDA and debugging)
- **`model_ready_{name}.csv`** — only the feature manifest columns + essential auxiliary columns (`log_price`, `price_numeric`, `zone`, `sub_zone`, `gouvernorat`, spatial columns, `description`) needed by the modelling notebook

Additional outputs:
- **`predict_pool.csv`** — 4,015 listings with null price (`"Prix à consulter"`), ready for inference once models are trained
- **`pipeline_report.txt`** — full audit trail of every cleaning step with row counts

### Pipeline summary
| Stage | Rows |
|---|---|
| Raw input | 30,186 |
| In trainable sub-datasets | 17,323 (57.4%) |
| Predict pool (null price) | 4,015 (13.3%) |
| Lost to cleaning | 8,848 (29.3%) |

In [ ]:
print('STEP 10: Save outputs')
output_dir = CONFIG['output_dir']
os.makedirs(output_dir, exist_ok=True)
saved_paths = []

AUX_COLS = [
    'log_price', 'price_numeric', 'adresse', 'datepost',
    'category', 'transaction', 'zone', 'sub_zone',
    'surface_numeric', 'log_surface', 'description',
    'lat', 'lon', 'has_coords',
    'dist_tunis_centre', 'dist_to_coast', 'dist_lac', 'dist_la_marsa',
    'dist_hammamet', 'dist_sousse', 'dist_sfax', 'dist_djerba',
    'gouvernorat', 'gouvernorat_cat',
]

for name, sub_df in datasets.items():
    feats = FEATURE_MANIFESTS[name]

    full_path = os.path.join(output_dir, f'clean_{name}.csv')
    sub_df.to_csv(full_path, index=False)
    saved_paths.append(full_path)

    keep    = list(dict.fromkeys(feats + [c for c in AUX_COLS if c in sub_df.columns]))
    mr_path = os.path.join(output_dir, f'model_ready_{name}.csv')
    sub_df[keep].to_csv(mr_path, index=False)
    saved_paths.append(mr_path)

    print(f'  clean_{name}.csv       -> {len(sub_df):,} rows, {sub_df.shape[1]} cols')
    print(f'  model_ready_{name}.csv -> {len(feats)} features + {len(keep)-len(feats)} aux cols')

pool_path = os.path.join(output_dir, 'predict_pool.csv')
predict_pool.to_csv(pool_path, index=False)
saved_paths.append(pool_path)
print(f'  predict_pool.csv       -> {len(predict_pool):,} rows (null-price listings for future inference)')

report_path = os.path.join(output_dir, 'pipeline_report.txt')
with open(report_path, 'w') as f:
    f.write('Kadastra BO2 — Preprocessing Pipeline Report\n')
    f.write(f'Generated: {datetime.now().isoformat()}\n')
    f.write('=' * 60 + '\n\n')
    for entry in audit:
        f.write(f'[{entry["step"]}]\n')
        f.write(f'  {entry["rows_before"]:,} -> {entry["rows_after"]:,}'
                f'  (dropped {entry["dropped"]:,} = {entry["dropped_pct"]}%)\n')
        if entry['detail']:
            f.write(f'  {entry["detail"]}\n')
        f.write('\n')
    f.write('\nSub-dataset summary:\n')
    for name, sub_df in datasets.items():
        f.write(f'  {name}: {len(sub_df):,} rows, {len(FEATURE_MANIFESTS[name])} features\n')
    n_coords_final = df['has_coords'].sum()
    f.write(f'\nRows with valid coordinates: {n_coords_final:,}/{len(df):,} '
            f'({n_coords_final/len(df)*100:.1f}%)\n')
saved_paths.append(report_path)

total_end = sum(len(d) for d in datasets.values())
print(f'\n{"="*50}')
print(f'PIPELINE COMPLETE')
print(f'  Started with       : {TOTAL_START:,} rows')
print(f'  Trainable datasets : {total_end:,} rows ({total_end/TOTAL_START*100:.1f}%)')
print(f'  Predict pool       : {len(predict_pool):,} rows')
print(f'  Lost to cleaning   : {TOTAL_START - total_end - len(predict_pool):,} rows')
print(f'{"="*50}')

print('\nDownloading all output files...')
for path in saved_paths:
    files.download(path)
    print(f'  downloaded: {os.path.basename(path)}')

STEP 10: Save outputs
  clean_appt_vente.csv       -> 4,383 rows, 82 cols
  model_ready_appt_vente.csv -> 57 features + 10 aux cols
  clean_appt_location.csv       -> 7,250 rows, 82 cols
  model_ready_appt_location.csv -> 58 features + 10 aux cols
  clean_maison_vente.csv       -> 2,636 rows, 82 cols
  model_ready_maison_vente.csv -> 54 features + 10 aux cols
  clean_maison_location.csv       -> 1,105 rows, 82 cols
  model_ready_maison_location.csv -> 53 features + 10 aux cols
  clean_terrain_vente.csv       -> 1,509 rows, 82 cols
  model_ready_terrain_vente.csv -> 41 features + 10 aux cols
  clean_local_vente.csv       -> 440 rows, 82 cols
  model_ready_local_vente.csv -> 39 features + 16 aux cols
  predict_pool.csv       -> 4,015 rows (null-price listings for future inference)

PIPELINE COMPLETE
  Started with       : 30,186 rows
  Trainable datasets : 17,323 rows (57.4%)
  Predict pool       : 4,015 rows
  Lost to cleaning   : 8,848 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  downloaded: clean_appt_vente.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  downloaded: predict_pool.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  downloaded: pipeline_report.txt
